In [1]:
#Gauntlet #5: Combine Fragmented Titanic Data
"""You are given the Titanic dataset broken into four separate fragments with overlapping and 
missing information. Your mission is to reconstruct a single, clean DataFrame containing 
all passengers with complete information where possible."""

'You are given the Titanic dataset broken into four separate fragments with overlapping and \nmissing information. Your mission is to reconstruct a single, clean DataFrame containing \nall passengers with complete information where possible.'

In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv('titanic.csv')

# Fragment 1: Passenger details (PassengerId 1-300)
frag1 = df[df['PassengerId'] <= 300][['PassengerId', 'Name', 'Sex', 'Age']].copy()

# Fragment 2: Ticket info (PassengerId 250-550)
frag2 = df[(df['PassengerId'] >= 250) & (df['PassengerId'] <= 550)][['PassengerId', 'Pclass', 'Fare', 'Embarked']].copy()

# Fragment 3: Family info (PassengerId 500-891)
frag3 = df[df['PassengerId'] >= 500][['PassengerId', 'SibSp', 'Parch']].copy()

# Fragment 4: Survival info for all passengers (PassengerId 1-891)
frag4 = df[['PassengerId', 'Survived']].copy()

In [5]:
# Fragment 1b: Backup for IDs 301–400 (simulated additional fragment)
frag1b = df[(df['PassengerId'] >= 301) & (df['PassengerId'] <= 400)][['PassengerId', 'Name', 'Sex', 'Age']].copy()

# Task 1: Vertical concatenation
passengers_full = pd.concat([frag1, frag1b], axis=0, ignore_index=True)
print(f"Task 1 - passengers_full shape: {passengers_full.shape}")
print(passengers_full.head())

Task 1 - passengers_full shape: (400, 4)
   PassengerId                                               Name     Sex  \
0            1                            Braund, Mr. Owen Harris    male   
1            2  Cumings, Mrs. John Bradley (Florence Briggs Th...  female   
2            3                             Heikkinen, Miss. Laina  female   
3            4       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female   
4            5                           Allen, Mr. William Henry    male   

    Age  
0  22.0  
1  38.0  
2  26.0  
3  35.0  
4  35.0  


In [6]:
# Fragment 2: Ticket info (IDs 250–550)
frag2 = df[(df['PassengerId'] >= 250) & (df['PassengerId'] <= 550)][['PassengerId', 'Pclass', 'Fare', 'Embarked']].copy()

# Task 2: Outer merge with indicator and custom suffixes
merged_1 = pd.merge(
    passengers_full, 
    frag2, 
    on='PassengerId', 
    how='outer', 
    indicator=True,
    suffixes=('_Pass', '_Ticket')
)

print("\nTask 2 - Merge indicator counts:")
print(merged_1['_merge'].value_counts())


Task 2 - Merge indicator counts:
_merge
left_only     249
both          151
right_only    150
Name: count, dtype: int64


In [7]:
# Task 3: Drop redundant columns (any ending with '_Ticket' that are duplicates)
# In our case, only columns from frag2 have '_Ticket' suffix, but none are redundant with frag1.
# However, the task expects us to drop any redundant columns.
# We check columns:
print("\nTask 3 - Columns before cleaning:", merged_1.columns.tolist())

# Since there are no actual duplicate base names (only 'PassengerId' is common but handled),
# we simply note that no further drop is needed. If there were a column like 'Name_Ticket',
# we would drop it. We'll just keep the DataFrame as is.


Task 3 - Columns before cleaning: ['PassengerId', 'Name', 'Sex', 'Age', 'Pclass', 'Fare', 'Embarked', '_merge']


In [8]:
# Fragment 3: Family info (IDs 500–891)
frag3 = df[df['PassengerId'] >= 500][['PassengerId', 'SibSp', 'Parch']].copy()

# Task 4: Set index and left join
merged_1_indexed = merged_1.set_index('PassengerId')
frag3_indexed = frag3.set_index('PassengerId')

merged_2 = merged_1_indexed.join(frag3_indexed, how='left')
merged_2 = merged_2.reset_index()
print("\nTask 4 - merged_2 shape:", merged_2.shape)
print(merged_2.head())


Task 4 - merged_2 shape: (550, 10)
   PassengerId                                               Name     Sex  \
0            1                            Braund, Mr. Owen Harris    male   
1            2  Cumings, Mrs. John Bradley (Florence Briggs Th...  female   
2            3                             Heikkinen, Miss. Laina  female   
3            4       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female   
4            5                           Allen, Mr. William Henry    male   

    Age  Pclass  Fare Embarked     _merge  SibSp  Parch  
0  22.0     NaN   NaN      NaN  left_only    NaN    NaN  
1  38.0     NaN   NaN      NaN  left_only    NaN    NaN  
2  26.0     NaN   NaN      NaN  left_only    NaN    NaN  
3  35.0     NaN   NaN      NaN  left_only    NaN    NaN  
4  35.0     NaN   NaN      NaN  left_only    NaN    NaN  


In [9]:
# Task 5: Create age_backup (median age per Pclass)
# First, compute median age per class from original df
median_age_per_class = df.groupby('Pclass')['Age'].median()

# Create age_backup DataFrame for all passengers
age_backup = df[['PassengerId', 'Pclass']].copy()
age_backup['Age_Filled'] = age_backup['Pclass'].map(median_age_per_class)

# Merge to get Pclass for each passenger in merged_2
# (merged_2 already has Pclass from frag2, but some passengers may not have ticket info)
# We'll do a quick left merge with age_backup just to get Age_Filled aligned.
temp_for_age = pd.merge(
    merged_2[['PassengerId']], 
    age_backup[['PassengerId', 'Age_Filled']], 
    on='PassengerId', 
    how='left'
)

# Use combine_first on the Age column
merged_2['Age'] = merged_2['Age'].combine_first(temp_for_age['Age_Filled'])

print("\nTask 5 - Missing Age after combine_first:", merged_2['Age'].isna().sum())


Task 5 - Missing Age after combine_first: 0


In [10]:
# Fragment 4: Survival info for all passengers
frag4 = df[['PassengerId', 'Survived']].copy()

# Task 6: Inner merge to keep only passengers with survival data
final_df = pd.merge(merged_2, frag4, on='PassengerId', how='inner')

print("\nTask 6 - final_df shape:", final_df.shape)


Task 6 - final_df shape: (550, 11)


In [11]:
# Task 7: Verification
print("\nTask 7 - Verification:")
print(f"Final shape: {final_df.shape}")
print(f"Missing Age: {final_df['Age'].isna().sum()}")
print(f"Missing Embarked: {final_df['Embarked'].isna().sum()}")

# Optional: Show final_df info
print(final_df.info())


Task 7 - Verification:
Final shape: (550, 11)
Missing Age: 0
Missing Embarked: 249
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 550 entries, 0 to 549
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   PassengerId  550 non-null    int64   
 1   Name         400 non-null    object  
 2   Sex          400 non-null    object  
 3   Age          550 non-null    float64 
 4   Pclass       301 non-null    float64 
 5   Fare         301 non-null    float64 
 6   Embarked     301 non-null    object  
 7   _merge       550 non-null    category
 8   SibSp        51 non-null     float64 
 9   Parch        51 non-null     float64 
 10  Survived     550 non-null    int64   
dtypes: category(1), float64(5), int64(2), object(3)
memory usage: 43.8+ KB
None
